# 01 — Extract and validate two persona vectors (`evil`, `dishonest`)

Persona Vectors (Chen et al. 2025) §2–3, on Qwen3-4B. Run the notebook **once per trait** (`TRAIT` in the config cell).

**Pipeline**
1. **Rollouts.** 20 extraction questions × 5 contrastive system-prompt pairs (trait vs anti-trait) × N samples, sampled at T = 1.
2. **Judge.** Every response gets a trait score and a coherence score (0–100) from Claude.
3. **Filter** *(your gear)*: which rollouts count as clean on-trait / off-trait examples.
4. **Activations** *(your gear: the token reduction)*: residual stream at every block, collapsed over tokens.
5. **Vectors** *(your gear)*: one candidate direction per block.
6. **Steering sweep.** Add α·v at each layer while answering the 20 held-out eval questions; judge trait and coherence.
7. **Pick the layer** *(your gear)*, then a finer dose-response curve at that layer.
8. **Monitoring.** Does the projection at the last prompt token predict how much trait the response will show? (8-rung system-prompt ladder, §3.3.)
9. **Save** the vector for Exp 2 and for emergent-misalignment.

**Read first:** §2, §3, App. A.1–A.3, B.1, B.4, C.2–C.3.

**Where this differs from the paper**
- Model: Qwen3-4B (36 blocks, thinking off) instead of Qwen2.5-7B-Instruct / Llama-3.1-8B.
- Judge: Claude writing one integer, instead of GPT-4.1-mini with a logprob-weighted score. Scores come out coarser.
- `dishonest` is not a paper trait. Its artifacts were written by Claude with the paper's A.1 template; review `data/traits/dishonest.json` before running it.
- Fewer samples in the sweep and monitoring runs, to keep cost down. The knobs are in the config cells.
- Layers are 0-indexed blocks (repo convention), so the paper's "layer 20" is our 19.

**Compute:** needs a GPU (A100-class). A full run per trait is ≈ 2,600 generations and ≈ 4,500 judge calls; the judge prints its running cost. `SMOKE = True` is a tiny mechanics-only pass (Qwen3-0.6B, a handful of rollouts) that checks the plumbing and nothing else: its numbers mean nothing.

## Predictions — write before running

_Peter:_
- Best steering layer (0-indexed block of 36), evil / dishonest:
- Does dishonest steer as cleanly as evil (trait score at matched coherence)?
- Fraction of rollouts that survive your filter, pos / neg, per trait:
- Monitoring r, overall / within-level, evil / dishonest:
- What you expect to break:

_Claude:_ ask after you've written yours, so I don't anchor you.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import pv
import judge as J

TRAIT = "evil"   # "evil" | "dishonest": run the notebook once per trait
SMOKE = False    # True → tiny mechanics-only pass; numbers meaningless
SEED = 0

if SMOKE:
    MODEL_ID, N_QUESTIONS, N_PAIRS, N_PER_PAIR, MAX_NEW = "Qwen/Qwen3-0.6B", 2, 1, 1, 48
else:
    MODEL_ID, N_QUESTIONS, N_PAIRS, N_PER_PAIR, MAX_NEW = None, None, None, 2, 384

GEN = dict(max_new_tokens=MAX_NEW, batch_size=32, seed=SEED)
TAG = f"{TRAIT}{'_smoke' if SMOKE else ''}"   # cache-file prefix

trait = pv.load_trait(TRAIT)
judge = J.Judge()   # J.Judge(model="claude-haiku-5-5") costs ~40x less
model, tok, device = pv.load_model(MODEL_ID)
print(model.config._name_or_path, "| blocks:", pv.n_blocks(model), "| device:", device, "| judge:", judge.model)

In [ ]:
# What the model actually sees, for pair 0
print(pv.system_prompt(trait, 0, "pos"))
print(pv.system_prompt(trait, 0, "neg"))
print(repr(pv.chat_prompt(tok, pv.system_prompt(trait, 0, "pos"), trait["questions_extract"][0])))

## 1. Rollouts

In [ ]:
table = pv.rollout_table(trait, "extract", n_per_pair=N_PER_PAIR, n_questions=N_QUESTIONS, n_pairs=N_PAIRS)
print(len(table), "rollouts")
roll = pv.cached_df(f"rollouts_{TAG}", lambda: pv.generate_rollouts(model, tok, table, device=device, **GEN))
print(f"truncated: {roll.truncated.mean():.1%}   mean tokens: {roll.n_tokens.mean():.0f}")
roll[["pair", "polarity", "question", "response"]].head(4)

## 2. Judge

Cached per call in `cache/judge_cache.jsonl`, so re-running this cell is free.

In [ ]:
scored = judge.score_df(roll, trait["eval_prompt"])
scored.groupby("polarity")[["trait_score", "coherence", "refusal", "truncated"]].mean().round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, col in zip(axes, ["trait_score", "coherence"]):
    for pol, c in [("pos", "C3"), ("neg", "C0")]:
        ax.hist(scored.loc[scored.polarity == pol, col].dropna(), bins=np.arange(0, 105, 5),
                alpha=.6, color=c, label=pol)
    ax.set_title(col); ax.legend()
plt.tight_layout()

# Which instruction pairs actually elicit the trait?
display(scored.pivot_table(index="pair", columns="polarity", values="trait_score").round(1))

for pol in ("pos", "neg"):
    r = scored[scored.polarity == pol].sample(1, random_state=SEED).iloc[0]
    print(f"--- {pol} | trait {r.trait_score} | coherence {r.coherence}\nQ: {r.question}\nA: {r.response[:600]}\n")

## 3. Filter — your gear

`pv.filter_rollouts`: what counts as a clean on-trait / off-trait example? Keep the original index: step 5 uses it to pick the matching activations.

In [ ]:
kept = pv.filter_rollouts(scored)
print(kept.polarity.value_counts().to_dict(), f"kept of {len(scored)}")
kept.groupby("qid").polarity.value_counts().unstack(fill_value=0).T   # coverage per question

## 4. Activations — your gear: `pv.reduce_tokens`

Each rollout is re-run once with every block's residual stream, and your `reduce_tokens(h, n_prompt)` collapses the token axis. This runs over **all** rollouts, not just the kept ones, so changing the filter doesn't invalidate the cache. That means your reduce has to tolerate a response with zero tokens (`n_prompt == seq`). Bump `REDUCE_TAG` whenever you change `reduce_tokens`.

In [ ]:
REDUCE_TAG = "v1"
name = f"acts_{TAG}_{REDUCE_TAG}"
if (pv.CACHE_DIR / f"{name}.pt").exists():
    acts_all = pv.load_tensor(name)
else:
    acts_all = pv.collect_reduced(model, tok, scored.prompt.tolist(), scored.response.tolist(),
                                  pv.reduce_tokens, device=device)
    pv.save_tensor(acts_all, name)
print(acts_all.shape)   # (n rollouts, n_blocks, d)

## 5. Persona vectors — your gear: `pv.persona_vectors`

In [ ]:
acts = acts_all[torch.as_tensor(kept.index.values)]
V = pv.persona_vectors(acts, kept.reset_index(drop=True))   # (n_blocks, d)
assert V.shape == acts.shape[1:], V.shape

vnorm = V.norm(dim=-1)
resid_norm = acts_all.norm(dim=-1).mean(0)   # typical norm of a reduced activation, per block
Vn = V / vnorm[:, None]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(vnorm, label="‖v_L‖")
axes[0].plot(resid_norm, label="mean ‖reduced h_L‖")
axes[0].set_yscale("log"); axes[0].set_xlabel("block"); axes[0].legend()
im = axes[1].imshow((Vn @ Vn.T).numpy(), cmap="RdBu_r", vmin=-1, vmax=1)
axes[1].set_title("cos(v_L, v_L')"); axes[1].set_xlabel("block"); plt.colorbar(im, ax=axes[1])
plt.tight_layout()

## 6. Steering sweep across layers (Fig. 3, App. B.4)

Same coefficients at every layer, on the 20 held-out eval questions, no system prompt. Set `SWEEP_COEFFS` in the units your normalization from step 5 implies: the norm plot above gives the scale.

In [ ]:
SWEEP_LAYERS = [1, 3] if SMOKE else list(range(6, 33, 3))   # 0-indexed blocks
SWEEP_COEFFS = None   # set: ~3 values, in the units of V
N_SWEEP = 1 if SMOKE else 2
assert SWEEP_COEFFS, "set SWEEP_COEFFS"

eval_qs = trait["questions_eval"][:N_QUESTIONS]
sweep = pv.cached_df(
    f"sweep_{TAG}_{pv.key(V, SWEEP_LAYERS, SWEEP_COEFFS, N_SWEEP, MAX_NEW)}",
    lambda: pv.steering_sweep(model, tok, eval_qs, V, SWEEP_LAYERS, SWEEP_COEFFS,
                              n_samples=N_SWEEP, device=device, **GEN))
sweep = judge.score_df(sweep, trait["eval_prompt"])

In [ ]:
base = sweep[sweep.layer == -1][["trait_score", "coherence"]].mean()
g = sweep[sweep.layer >= 0].groupby(["coeff", "layer"])[["trait_score", "coherence"]].mean().reset_index()
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharex=True)
for ax, col in zip(axes, ["trait_score", "coherence"]):
    for c, d in g.groupby("coeff"):
        ax.plot(d.layer, d[col], marker="o", label=f"α = {c:g}")
    ax.axhline(base[col], color="k", ls="--", lw=1, label="unsteered")
    ax.set_xlabel("block"); ax.set_title(col); ax.set_ylim(-2, 102)
axes[0].legend(); plt.tight_layout()

## 7. Pick the layer — your gear

Write the rule and the reason here before setting `LAYER`.

_Rule:_

In [ ]:
LAYER = None   # TODO_PETER: 0-indexed block, from the sweep, by the rule written above
assert LAYER is not None
v = V[LAYER]

### Dose-response at the chosen layer

In [ ]:
DOSE_COEFFS = None   # set: ~6 values from "no visible effect" to "incoherent", same units as SWEEP_COEFFS
N_DOSE = 1 if SMOKE else 3
assert DOSE_COEFFS, "set DOSE_COEFFS"

dose = pv.cached_df(
    f"dose_{TAG}_{pv.key(v, LAYER, DOSE_COEFFS, N_DOSE, MAX_NEW)}",
    lambda: pv.steering_sweep(model, tok, eval_qs, V, [LAYER], DOSE_COEFFS,
                              n_samples=N_DOSE, device=device, **GEN))
dose = judge.score_df(dose, trait["eval_prompt"])

d = dose.groupby("coeff")[["trait_score", "coherence"]].agg(["mean", "sem"])
fig, ax = plt.subplots(figsize=(6, 3.5))
for col, c in [("trait_score", "C3"), ("coherence", "C0")]:
    ax.errorbar(d.index, d[(col, "mean")], yerr=d[(col, "sem")], marker="o", color=c, label=col, capsize=3)
ax.set_xlabel(f"α at block {LAYER}"); ax.set_ylim(-2, 102); ax.legend()
plt.tight_layout()

# Read a few: strongest coefficient that still reads as coherent
top = dose[(dose.coherence >= 50) & (dose.coeff > 0)].sort_values(["coeff", "trait_score"]).tail(2)
for _, r in top.iterrows():
    print(f"--- α {r.coeff:g} | trait {r.trait_score} | coherence {r.coherence}\nQ: {r.question}\nA: {r.response[:600]}\n")

## 8. Monitoring (§3.3, Fig. 4, App. C.2)

8 system prompts from trait-discouraging (level 0) to trait-encouraging (level 7), × the 20 eval questions. Project the **last prompt token** at `LAYER` onto v̂ (before any response exists), then judge what the model actually says. The question: does the projection predict the behaviour, overall and *within* one system prompt?

In [ ]:
ladder = pv.load_ladder(TRAIT)[:: 7 if SMOKE else 1]   # smoke: just the two ends
N_MON = 1 if SMOKE else 4

mon = pv.cached_df(
    f"monitor_{TAG}_{pv.key(v, LAYER, ladder, N_MON, MAX_NEW)}",
    lambda: pv.monitoring_run(model, tok, ladder, eval_qs, LAYER, v, n_samples=N_MON, device=device, **GEN))
mon = judge.score_df(mon, trait["eval_prompt"], coherence=False)
agg, r = pv.monitor_correlations(mon)
print(r)

fig, ax = plt.subplots(figsize=(6, 4))
sc = ax.scatter(agg.projection, agg.trait_score, c=agg.level, cmap="viridis", s=18)
plt.colorbar(sc, label="ladder level (0 = discourages trait)")
ax.set_xlabel(f"projection onto v̂, block {LAYER}, last prompt token")
ax.set_ylabel(f"mean {TRAIT} score")
ax.set_title(f"overall r = {r['overall_r']:.2f}   within-level r = {r['within_level_r']:.2f}")
plt.tight_layout()

## 9. Save

In [ ]:
if not SMOKE:
    p = pv.save_persona(TRAIT, V, LAYER, reduce_tag=REDUCE_TAG, n_kept=len(kept),
                        judge=judge.model, monitor=r, model=model.config._name_or_path)
    print("saved", p)
print(f"judge cost this session ≈ ${judge.cost():.2f}")

## Results

- **evil:** layer __; steering __; monitoring r overall __ / within __
- **dishonest:** layer __; steering __; monitoring r overall __ / within __
- **vs predictions:**
- **Parked** (one line, only if something comes up):